# Regression with scikit-learn

<a target="_blank" href="https://colab.research.google.com/github/imamitjain/notebooks/blob/main/02-ml-fundamentals/01_sklearn_regression.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Objective:** Understand the regression workflow end-to-end — from data preparation through model training, evaluation, and interpretation using scikit-learn.

**Prerequisites:** NumPy, Pandas, Matplotlib (Section 01)

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q numpy pandas matplotlib scikit-learn


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.datasets import make_regression

## 1. Generate Synthetic Data

In [ ]:
X, y = make_regression(n_samples=200, n_features=1, noise=15, random_state=42)

plt.figure(figsize=(8, 5))
plt.scatter(X, y, alpha=0.6, edgecolors='k', linewidth=0.5)
plt.xlabel("Feature")
plt.ylabel("Target")
plt.title("Synthetic Regression Data")
plt.show()

## 2. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")

## 3. Linear Regression

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f"Coefficient: {model.coef_[0]:.2f}")
print(f"Intercept:   {model.intercept_:.2f}")
print(f"R² score:    {r2_score(y_test, y_pred):.4f}")
print(f"RMSE:        {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(X_test, y_test, alpha=0.6, label="Actual", edgecolors='k', linewidth=0.5)

X_line = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
plt.plot(X_line, model.predict(X_line), color='red', linewidth=2, label="Prediction")

plt.xlabel("Feature")
plt.ylabel("Target")
plt.title("Linear Regression Fit")
plt.legend()
plt.show()

## 4. Regularization — Ridge and Lasso

In [ ]:
X_multi, y_multi = make_regression(n_samples=200, n_features=20, n_informative=5, noise=10, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_multi, y_multi, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr)
X_te_scaled = scaler.transform(X_te)

results = {}
for name, model in [("Linear", LinearRegression()),
                     ("Ridge (α=1)", Ridge(alpha=1.0)),
                     ("Lasso (α=1)", Lasso(alpha=1.0))]:
    model.fit(X_tr_scaled, y_tr)
    y_p = model.predict(X_te_scaled)
    results[name] = {
        "R²": r2_score(y_te, y_p),
        "RMSE": np.sqrt(mean_squared_error(y_te, y_p)),
        "Non-zero coefs": np.sum(model.coef_ != 0)
    }

pd.DataFrame(results).T

## 5. Residual Analysis

In [ ]:
model_final = Ridge(alpha=1.0)
model_final.fit(X_tr_scaled, y_tr)
y_pred_final = model_final.predict(X_te_scaled)
residuals = y_te - y_pred_final

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(y_pred_final, residuals, alpha=0.6, edgecolors='k', linewidth=0.5)
axes[0].axhline(y=0, color='red', linestyle='--')
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Residuals")
axes[0].set_title("Residuals vs Predicted")

axes[1].hist(residuals, bins=20, edgecolor='black', alpha=0.7)
axes[1].set_xlabel("Residual Value")
axes[1].set_title("Residual Distribution")

plt.tight_layout()
plt.show()

## Try It Yourself

1. Load the Boston-style housing dataset (`sklearn.datasets.fetch_california_housing`). Build a Ridge regression, tune `alpha` using a loop over `[0.01, 0.1, 1, 10, 100]`, and plot R² vs alpha.
2. Add polynomial features (`sklearn.preprocessing.PolynomialFeatures`) to the 1D synthetic data and compare linear vs polynomial regression.
3. Implement a simple gradient descent for linear regression from scratch using NumPy and compare the learned coefficients to sklearn's result.

In [ ]:
# Your code here